## Lendo fonte de dados

In [0]:
%python

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Workspace/Users/jhenivon@hotmail.com/Projetos/dados/combustivel/*.csv", sep=";")


In [0]:
%python
print("Qtde de Linhas de dados", df_raw.count() )
display(df_raw.limit(5) )

Qtde de Linhas de dados 813731


Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
NE,CE,SOBRAL,ECONOGÁS DO BRASIL DIST. DERIV. DE PET. BIOC. E GÁS NAT,08.775.979/0002-62,RUA TABELIÃO IDELFONSO CAVALCANTI,455,null,CENTRO,62010-000,GASOLINA,2025-01-01,"6,29",null,R$ / litro,RAIZEN
NE,CE,SOBRAL,ECONOGÁS DO BRASIL DIST. DERIV. DE PET. BIOC. E GÁS NAT,08.775.979/0002-62,RUA TABELIÃO IDELFONSO CAVALCANTI,455,null,CENTRO,62010-000,GASOLINA ADITIVADA,2025-01-01,"6,49",null,R$ / litro,RAIZEN
NE,CE,SOBRAL,ECONOGÁS DO BRASIL DIST. DERIV. DE PET. BIOC. E GÁS NAT,08.775.979/0002-62,RUA TABELIÃO IDELFONSO CAVALCANTI,455,null,CENTRO,62010-000,DIESEL S10,2025-01-01,"6,19",null,R$ / litro,RAIZEN
NE,CE,SOBRAL,ECONOGÁS DO BRASIL DIST. DERIV. DE PET. BIOC. E GÁS NAT,08.775.979/0002-62,RUA TABELIÃO IDELFONSO CAVALCANTI,455,null,CENTRO,62010-000,ETANOL,2025-01-01,"5,19",null,R$ / litro,RAIZEN
NE,CE,SOBRAL,V.C.EMPREENDIMENTOS LTDA,03.551.935/0002-35,AVENIDA JOSE EUCLIDES FERREIRA GOMES,30,POSTO FLASH,CORACAO DE JESUS,62043-070,GASOLINA,2025-01-01,"6,53",null,R$ / litro,RAIZEN


## Camada Bronze

Criando tbl bronze_combustivel

In [0]:
%python
df_raw.createOrReplaceTempView("bronze_combustivel")

## Criando camada Silver SQL
Criando tabela Silver com suas alterações e padronização de dados

In [0]:
CREATE OR REPLACE TEMP VIEW silver_combustivel AS
SELECT DISTINCT
    `Regiao - Sigla`       AS Regiao_Sigla,
    `Estado - Sigla`       AS Estado_Sigla,
    Municipio,
    Revenda,
    `CNPJ da Revenda`      AS Cnpj_Revenda,
    Bairro,
    Produto,
    `Data da Coleta`       AS Data_Coleta,

    CAST(
        REPLACE(`Valor de Venda`, ',', '.')
        AS DECIMAL(10,2)
    ) AS Valor_Venda,

    CASE
        WHEN `Unidade de Medida` = 'R$ / m3'
        THEN 'R$ / m³'
        ELSE `Unidade de Medida`
    END AS Unidade_Medida,
    Bandeira

FROM bronze_combustivel
WHERE Produto IS NOT NULL;

### Minha checklist final de Silver
* ✅ Contagem Bronze x Silver
* ✅ Nulos tratados
* ✅ Duplicados removidos
* ✅ Unidade de medida padronizada
* ✅ Tipo Valor_Venda convertido para Decimal
* ✅ Datas válidas
* ✅ Valores monetários válidos
* ✅ Produtos válidos

## Camada Gold

 Cria camada gold_combustivel enriquecida para Análise de Negócios

 O comando DDL + CTAS (Create Table As Select).

 Mas ele não retorna linhas para exibição(Mesmo assim as linha são inseridas)

In [0]:
CREATE OR REPLACE TABLE gold_combustivel AS
SELECT 
    Regiao_Sigla,
    Estado_Sigla,
    Municipio,
    Revenda,
    Cnpj_Revenda,
    Bairro,
    Produto,
    Data_Coleta,
    YEAR(Data_Coleta) AS Ano,
    MONTH(Data_Coleta) AS Mes,
    --DATE_TRUNC('month', Data_Coleta) AS Ano_Mes,
    date_format(Data_Coleta, 'yyyy-MM') AS Ano_Mes,
    Valor_Venda,
    Unidade_Medida,
    Bandeira

    FROM silver_combustivel;

num_affected_rows,num_inserted_rows


## Validando Qtde de Linhas e amostra de dados

In [0]:
SELECT COUNT(*) DISTINCT FROM gold_combustivel;

DISTINCT
804617


In [0]:
SELECT * FROM gold_combustivel LIMIT 5;

Regiao_Sigla,Estado_Sigla,Municipio,Revenda,Cnpj_Revenda,Bairro,Produto,Data_Coleta,Ano,Mes,Ano_Mes,Valor_Venda,Unidade_Medida,Bandeira
SE,SP,JACAREI,POSTO DE SERVICOS SHOPPING DE JACAREI LTDA,62.122.908/0001-25,VILA PINHEIRO,ETANOL,2025-01-01,2025,1,2025-01,3.99,R$ / litro,IPIRANGA
SE,SP,SAO VICENTE,AUTO POSTO MAR PEQUENO DE SAO VICENTE LTDA,09.165.024/0001-75,JAPUI,DIESEL S10,2025-01-01,2025,1,2025-01,6.19,R$ / litro,VIBRA
NE,RN,CAICO,CENTRO COMERCIAL DE COMBUSTIVEIS DOMINGOS LTDA,45.338.786/0001-90,CENTRO,ETANOL,2025-01-01,2025,1,2025-01,5.69,R$ / litro,VIBRA
N,TO,PARAISO DO TOCANTINS,MEDEIROS E CIA LTDA,04.874.758/0001-00,CENTRO,GASOLINA ADITIVADA,2025-01-02,2025,1,2025-01,6.25,R$ / litro,BRANCA
S,PR,ARAUCARIA,AUTO POSTO FIALLA LTDA,78.951.779/0001-18,CENTRO,DIESEL S10,2025-01-02,2025,1,2025-01,5.99,R$ / litro,BRANCA


# Fim